# 🎬 GAPT-CINEMA: Faststart & GPU Optimizer (Google Colab)
สมุดบันทึกสำหรับแปลงโค้ดวิดีโอ (เช่น **AV1 / HEVC** ให้เป็น **H.264 + AAC + Faststart**) เพื่อให้เล่นบน **Smart TV**, เบราว์เซอร์ และอุปกรณ์ทุกชนิดได้ทันที 100% ไม่สะดุด ไม่ติดจอดำ

### ⚡ วิธีใช้งานง่าย ๆ แค่ 1 คลิก:
1. เลือกเมนูด้านบน: **Runtime (รันไทม์)** → **Change runtime type (เปลี่ยนประเภทรันไทม์)** → ตรวจสอบว่าเลือก **T4 GPU** แล้ว
2. กด **Runtime (รันไทม์)** → **Run all (เรียกใช้ทั้งหมด)** หรือกดแป้นพิมพ์ **Ctrl + F9**
3. ระบบจะดาวน์โหลด แปลงไฟล์ด้วย GPU และส่งไฟล์ขึ้น Telegram พร้อมอัปเดตระบบสตรีมให้อัตโนมัติ!

### 1️⃣ ตรวจสอบสถานะ GPU และติดตั้ง Dependencies

In [ ]:
!nvidia-smi
!pip install -q telethon cryptg requests urllib3

# @title ⚙️ การตั้งค่าภาพยนตร์ (Movie Configuration)
# @markdown เลือกว่าจะดึงเรื่องที่กดสั่งจากเว็บ TGDrive / เรื่องล่าสุด หรือระบุ URL เอง
SELECT_MODE = "Auto (ตามที่สั่งจากเว็บ / เรื่องล่าสุด)" # @param ["Auto (ตามที่สั่งจากเว็บ / เรื่องล่าสุด)", "Manual URL"]
MANUAL_STREAM_URL = "https://stream.capt-th.work/stream/-1004381915212/13" # @param {type:"string"}
MANUAL_MOVIE_TITLE = "Three Thousand Years of Longing (2022).mp4" # @param {type:"string"}

import os, sys, time, json, subprocess, urllib.request

STREAM_URL = ""
MOVIE_TITLE = ""

if SELECT_MODE.startswith("Auto"):
    print("🔍 [1/2] ตรวจสอบคำสั่งแปลงไฟล์จากเว็บ TGDrive / GAPT-CINEMA...")
    got_target = False
    try:
        req = urllib.request.Request(
            "https://stream.capt-th.work/api/colab/target",
            headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
        )
        with urllib.request.urlopen(req, timeout=6) as resp:
            target = json.loads(resp.read().decode("utf-8"))
            if target.get("stream_url") and target.get("title"):
                STREAM_URL = target["stream_url"]
                MOVIE_TITLE = target["title"]
                got_target = True
                print(f"🎯 ได้รับคำสั่งแปลงไฟล์ที่กดมาจากเว็บ TGDrive เรียบร้อยแล้ว!")
                print(f"   - ชื่อภาพยนตร์: '{MOVIE_TITLE}'")
                print(f"   - Stream Source: {STREAM_URL}")
    except Exception as e:
        print(f"   (API target check note: {e})")

    if not got_target:
        print("🔍 [2/2] ตรวจสอบรายชื่อภาพยนตร์จาก GAPT-CINEMA API...")
        try:
            req = urllib.request.Request(
                "https://stream.capt-th.work/api/movies",
                headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
            )
            with urllib.request.urlopen(req, timeout=10) as resp:
                movies = json.loads(resp.read().decode("utf-8"))
            latest_movie = sorted(movies, key=lambda m: m.get("msg_id", 0), reverse=True)[0]
            STREAM_URL = f"https://stream.capt-th.work/stream/{latest_movie['chat_id']}/{latest_movie['msg_id']}"
            MOVIE_TITLE = latest_movie.get("file_name", "movie.mp4")
            print(f"🎯 ตรวจพบเรื่องล่าสุดอัตโนมัติ: '{MOVIE_TITLE}' (Msg ID: {latest_movie.get('msg_id')})")
            print(f"   - Stream Source: {STREAM_URL}")
        except Exception as e:
            print(f"⚠️ ใช้ค่าเริ่มต้น: {e}")
            STREAM_URL = MANUAL_STREAM_URL
            MOVIE_TITLE = MANUAL_MOVIE_TITLE
else:
    STREAM_URL = MANUAL_STREAM_URL.strip()
    MOVIE_TITLE = MANUAL_MOVIE_TITLE.strip()
    print(f"📌 ใช้งานการตั้งค่าแบบระบุเอง: '{MOVIE_TITLE}'")
    print(f"🔗 Stream Source: {STREAM_URL}")


In [ ]:
# @title ⚙️ การตั้งค่าภาพยนตร์ (Movie Configuration)
# @markdown เลือกว่าจะดึงเรื่องที่กดสั่งจากเว็บ TGDrive / เรื่องล่าสุดที่ยังไม่ได้แปลง หรือระบุ URL เอง
SELECT_MODE = "Auto (ตามที่สั่งจากเว็บ / เรื่องที่ยังไม่แปลง)" # @param ["Auto (ตามที่สั่งจากเว็บ / เรื่องที่ยังไม่แปลง)", "Manual URL"]
MANUAL_STREAM_URL = "https://stream.capt-th.work/stream/-1004381915212/13" # @param {type:"string"}
MANUAL_MOVIE_TITLE = "Three Thousand Years of Longing (2022).mp4" # @param {type:"string"}

import os, sys, time, json, subprocess, urllib.request

STREAM_URL = ""
MOVIE_TITLE = ""
ORIGINAL_MSG_ID = None
SKIP_TRANSCODE = False

# ดึงรายการที่เคยแปลงแล้ว (ขึ้นไอคอน ✅) เพื่อไม่ให้แปลงซ้ำเด็ดขาด
converted_ids = set()
try:
    req_c = urllib.request.Request("https://stream.capt-th.work/api/converted", headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req_c, timeout=6) as resp:
        c_data = json.loads(resp.read().decode("utf-8"))
        converted_ids = set(c_data if isinstance(c_data, list) else [])
    print(f"📋 รายการ Message ID ที่แปลงแล้วในระบบ (มีไอคอน ✅): {list(converted_ids)}")
except Exception as e:
    print(f"   (หมายเหตุการตรวจสอบ API: {e})")

if SELECT_MODE.startswith("Auto"):
    print("🔍 [1/2] ตรวจสอบคำสั่งแปลงไฟล์ที่ส่งมาจากเว็บ TGDrive...")
    got_target = False
    try:
        req = urllib.request.Request("https://stream.capt-th.work/api/colab/target", headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=6) as resp:
            target = json.loads(resp.read().decode("utf-8"))
            target_mid = target.get("msg_id")
            if target_mid and int(target_mid) in converted_ids:
                print(f"🛑 เรื่อง '{target.get('title')}' (ID: {target_mid}) เคยแปลงแล้ว (ขึ้นไอคอน ✅) ➔ ระบบจะไม่แปลงซ้ำ!")
                SKIP_TRANSCODE = True
                got_target = True
            elif target.get("stream_url") and target.get("title"):
                STREAM_URL = target["stream_url"]
                MOVIE_TITLE = target["title"]
                ORIGINAL_MSG_ID = target_mid
                got_target = True
                print(f"🎯 ได้รับคำสั่งแปลงไฟล์: '{MOVIE_TITLE}' (ID: {ORIGINAL_MSG_ID})")
    except Exception as e:
        pass

    if not got_target and not SKIP_TRANSCODE:
        print("🔍 [2/2] ตรวจสอบรายชื่อภาพยนตร์ที่ 'ยังไม่เคยแปลง' จาก GAPT-CINEMA...")
        try:
            req = urllib.request.Request("https://stream.capt-th.work/api/movies", headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=10) as resp:
                movies = json.loads(resp.read().decode("utf-8"))
            
            # กรองเฉพาะเรื่องที่ยังไม่ได้แปลง (ไม่อยู่ใน converted_ids)
            unconverted = [m for m in movies if m.get("msg_id") not in converted_ids and not m.get("faststart_ready")]
            if unconverted:
                latest_unconverted = sorted(unconverted, key=lambda m: m.get("msg_id", 0), reverse=True)[0]
                ORIGINAL_MSG_ID = latest_unconverted.get("msg_id")
                STREAM_URL = f"https://stream.capt-th.work/stream/{latest_unconverted['chat_id']}/{ORIGINAL_MSG_ID}"
                MOVIE_TITLE = latest_unconverted.get("file_name", "movie.mp4")
                print(f"🎯 ตรวจพบเรื่องที่ยังไม่ได้แปลง: '{MOVIE_TITLE}' (ID: {ORIGINAL_MSG_ID})")
                print(f"   - Stream Source: {STREAM_URL}")
            else:
                print("🎉 ทุกเรื่องในระบบขึ้นไอคอน ✅ (แปลงเรียบร้อยแล้วทั้งหมด 100%) ไม่ต้องแปลงเพิ่ม!")
                SKIP_TRANSCODE = True
        except Exception as e:
            print(f"⚠️ ใช้ค่าเริ่มต้น: {e}")
            STREAM_URL = MANUAL_STREAM_URL
            MOVIE_TITLE = MANUAL_MOVIE_TITLE
else:
    STREAM_URL = MANUAL_STREAM_URL.strip()
    MOVIE_TITLE = MANUAL_MOVIE_TITLE.strip()
    print(f"📌 ใช้งานการตั้งค่าแบบระบุเอง: '{MOVIE_TITLE}'")


### 3️⃣ ดาวน์โหลดต้นฉบับ & แปลงเป็น H.264 + Faststart ด้วย NVIDIA GPU

In [ ]:
if SKIP_TRANSCODE or not STREAM_URL:
    print("⏭️ ข้ามขั้นตอนการแปลงไฟล์: เนื่องจากเรื่องนี้เคยแปลงแล้ว (ขึ้นไอคอน ✅)")
else:
    INPUT_FILE = "input_raw.mp4"
    OUTPUT_FILE = "optimized_faststart.mp4"

    for f in [INPUT_FILE, OUTPUT_FILE]:
        if os.path.exists(f):
            os.remove(f)

    print(f"📥 [1/3] กำลังดาวน์โหลด '{MOVIE_TITLE}' จาก Cloudflare Stream Server...")
    t0 = time.time()
    !curl -L --retry 3 -H "User-Agent: Mozilla/5.0" -o {INPUT_FILE} "{STREAM_URL}"
    t1 = time.time()

    if not os.path.exists(INPUT_FILE) or os.path.getsize(INPUT_FILE) < 1024*1024:
        raise Exception("❌ ดาวน์โหลดไฟล์ไม่สำเร็จ กรุณาตรวจสอบ URL")

    in_bytes = os.path.getsize(INPUT_FILE)
    in_mb = in_bytes / (1024 * 1024)
    print(f"✅ ดาวน์โหลดเสร็จใน {t1 - t0:.1f} วินาที! ขนาด: {in_mb:.1f} MB ({in_bytes/(1024**3):.2f} GB)")

    print("🔍 ตรวจสอบ Codec ต้นฉบับด้วย ffprobe...")
    probe_cmd = [
        "ffprobe", "-v", "error", "-print_format", "json",
        "-show_format", "-show_streams", INPUT_FILE
    ]
    probe_res = subprocess.run(probe_cmd, capture_output=True, text=True)
    orig_duration = 0
    orig_w = 1920
    orig_h = 1080
    try:
        p_data = json.loads(probe_res.stdout)
        v_stream = next((s for s in p_data.get("streams", []) if s.get("codec_type") == "video"), {})
        orig_duration = int(float(p_data.get("format", {}).get("duration", 0)))
        orig_w = int(v_stream.get("width", 1920))
        orig_h = int(v_stream.get("height", 1080))
        print(f"   - Codec เดิม: {v_stream.get('codec_name', 'unknown')} | ขนาด: {orig_w}x{orig_h} | ความยาว: {orig_duration}s")
    except Exception as e:
        print(f"   - ffprobe info parse note: {e}")

    has_gpu = False
    try:
        res = subprocess.run(['nvidia-smi'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        if res.returncode == 0:
            has_gpu = True
    except Exception:
        pass

    print(f"⚡ [2/3] เริ่มแปลงเป็น H.264 + AAC + Faststart (GPU Mode: {has_gpu})...")
    if has_gpu:
        !ffmpeg -y -i {INPUT_FILE} \
          -c:v h264_nvenc -preset p4 -b:v 2200k -maxrate 2600k -bufsize 5000k \
          -c:a aac -b:a 128k \
          -movflags +faststart \
          {OUTPUT_FILE}
    else:
        !ffmpeg -y -i {INPUT_FILE} \
          -c:v libx264 -preset veryfast -b:v 2000k -maxrate 2400k -bufsize 4000k \
          -c:a aac -b:a 128k \
          -movflags +faststart \
          {OUTPUT_FILE}

    t2 = time.time()
    out_size = os.path.getsize(OUTPUT_FILE)
    out_mb = out_size / (1024 * 1024)
    print(f"\n🎉 แปลงไฟล์เสร็จสิ้นใน {t2 - t1:.1f} วินาที!")
    print(f"📦 ขนาดไฟล์ผลลัพธ์: {out_mb:.1f} MB ({out_size / (1024**3):.2f} GB)")
    print("✅ โครงสร้าง Faststart พร้อมเล่นบน Smart TV และเว็บได้ทันที 100%!")


### 4️⃣ ส่งไฟล์เข้า Telegram Channel และอัปเดตระบบสตรีมมิ่ง

In [ ]:
if SKIP_TRANSCODE or not STREAM_URL:
    print("⏭️ สิ้นสุดการทำงาน: ไม่มีการอัปโหลดไฟล์ซ้ำ เนื่องจากเรื่องนี้ขึ้นไอคอน ✅ เรียบร้อยแล้ว")
else:
    import asyncio, json, time, requests
    from telethon import TelegramClient
    from telethon.tl.types import DocumentAttributeVideo, DocumentAttributeFilename

    API_ID = 33156484
    API_HASH = "d23bc5cc206aeb76929cb3ff6481a5eb"
    BOT_TOKEN = "8627312320:AAGw9_C7okMMGMYj0VBRYfv80ppXBVtZdAo"
    CHANNEL_ID = -1004381915212
    FINAL_TITLE = MOVIE_TITLE
    if not FINAL_TITLE.lower().endswith(".mp4"):
        FINAL_TITLE += ".mp4"

    async def upload():
        print("🚀 กำลังเชื่อมต่อ Telegram Bot...")
        client = TelegramClient('colab_session', API_ID, API_HASH)
        await client.start(bot_token=BOT_TOKEN)
        try:
            out_size = os.path.getsize(OUTPUT_FILE)
            print(f"📤 กำลังส่งไฟล์ '{FINAL_TITLE}' ({out_size / (1024*1024):.1f} MB) เข้า Telegram...")
            
            def progress(current, total):
                pct = (current / total) * 100
                print(f"\rกำลังส่ง: {pct:.1f}% ({current/1024/1024:.1f}/{total/1024/1024:.1f} MB)", end="")

            msg = await client.send_file(
                CHANNEL_ID,
                OUTPUT_FILE,
                caption=FINAL_TITLE,
                supports_streaming=True,
                attributes=[
                    DocumentAttributeVideo(
                        duration=orig_duration if orig_duration > 0 else 5000,
                        w=orig_w,
                        h=orig_h,
                        supports_streaming=True
                    ),
                    DocumentAttributeFilename(file_name=FINAL_TITLE)
                ],
                progress_callback=progress
            )
            print(f"\n\n🎉 อัปโหลดสำเร็จแล้ว 100%!")
            print(f"📌 Message ID ใหม่บน Telegram: {msg.id}")
            
            # บันทึกสถานะว่าเรื่องเดิมและเรื่องใหม่ได้รับการแปลงแล้ว (ขึ้นไอคอน ✅)
            stream_url = f"https://stream.capt-th.work/watch/4381915212/{msg.id}"
            try:
                requests.get(stream_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=5)
                if ORIGINAL_MSG_ID:
                    requests.post("https://stream.capt-th.work/api/converted", json={"msg_id": int(ORIGINAL_MSG_ID)}, timeout=5)
                requests.post("https://stream.capt-th.work/api/converted", json={"msg_id": int(msg.id)}, timeout=5)
                print(f"✅ บันทึกสถานะแปลงสำเร็จ (ขึ้นไอคอน ✅) เรียบร้อย!")
            except Exception as e:
                print(f"   (หมายเหตุการบันทึกสถานะ: {e})")
            
            print(f"\n📺 ดูภาพยนตร์บน Smart TV และเว็บได้ทันที: {stream_url}")
        finally:
            await client.disconnect()

    await upload()
